# Stability at initialisation

How do normalization, input injection, and recurrent block depth affect randomly
initialised transformers? Compare each layer's trajectory with its final state,
then inspect the recurrent-state Jacobian and the input projection.

`mode="quick"` uses **one seed** and saves only in the run directory.
`mode="full"` uses **five seeds** and also updates the paper figures; SLURM selects
full mode. Both use the same architectures and **128-recurrence references**. Quick mode
reduces the number of initialisations, not model size or rollout length.

In [ ]:
mode = "quick"
layer_counts = [4, 8, 12, 16]  # Each depth × three norms × with/without injection.
recurrences = 128
summary_layers = 12
jacobian_layers = 1  # One-cycle Jacobian of a one-layer Pre Norm model.
diagnostic_layers = 6  # Ungated ReLU models for scalar measurements.
residual_scales = [0.0, 0.5]
residual_scale_recurrences = 256  # Last-step measurement for the ReLU diagnostic.
width = 512
num_heads = 4
sequence_length = 32
vocab_size = 32000
seed = 42
device = "auto"
output_dir = None

cache_dir = None  # Defaults to LOOPED_INTERNALS_CACHE_DIR or /data-gauss/hugbla/looped-internals-cache.
cache_mode = None  # Environment default; "reuse", "require" (no inference), or "off".
analysis_version = 1  # Bump when this notebook's scientific reduction changes.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import gc
from dataclasses import asdict
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from matplotlib.lines import Line2D
from torch.nn.attention import SDPBackend, sdpa_kernel
from tqdm.auto import tqdm

from experiments.cache import ExperimentCache
from experiments.support import (
    _set_seed,
    configure_experiment,
    export_paper_figures,
    get_figsize,
    save_figure,
    save_results,
)
from looped_internals.capture import compute_model_run_data
from looped_internals.models import MinimalRecurrentConfig, ModelSpec, load_model

plt.style.use(ROOT / "plots" / "matlab.mplstyle")
config = configure_experiment(
    experiment="initialisation",
    mode=mode,
    num_steps=recurrences,
    seed=seed,
    device=device,
    dtype="float32",
    output_dir=output_dir,
)
seeds = list(range(seed, seed + (1 if config.mode == "quick" else 5)))

result_cache = ExperimentCache(cache_dir, cache_mode)


def cached_measurement(measure, model_settings, run_seed, *inputs):
    key = result_cache.key(
        config,
        analysis_version=analysis_version,
        model_settings=asdict(model_settings),
        run_seed=run_seed,
        inputs=[value.tolist() for value in inputs],
    )
    return result_cache.run(
        f"initialisation/{measure.__name__}",
        key,
        lambda: measure(model_settings, run_seed, *inputs),
    )

In [ ]:
PRE_NORM = dict(
    norm_pre_attn=True,
    norm_post_attn=False,
    norm_pre_mlp=True,
    norm_post_mlp=False,
    norm_post_attn_residual=False,
    norm_post_mlp_residual=False,
    norm_post_cycle=False,
)
NORMALISATIONS = {
    "Pre": PRE_NORM,
    "Huginn-0125": PRE_NORM
    | dict(
        norm_post_attn_residual=True,
        norm_post_mlp_residual=True,
        norm_post_cycle=True,
    ),
    "Ouro": PRE_NORM
    | dict(
        norm_post_attn=True,
        norm_post_mlp=True,
        norm_post_cycle=True,
    ),
}
INJECTIONS = ["projection", None]


def model_config(
    layers, norm="Pre", injection=None, *, cycles=recurrences, **overrides
):
    return MinimalRecurrentConfig(
        d_model=width,
        n_heads=num_heads,
        n_layers=layers,
        num_cycles=cycles,
        d_vocab=vocab_size,
        input_injection=injection,
        **NORMALISATIONS[norm],
        **overrides,
    )


stability_configs = {
    (layers, norm, injection): model_config(layers, norm, injection)
    for layers in layer_counts
    for injection in INJECTIONS
    for norm in NORMALISATIONS
}
jacobian_configs = {
    injection: model_config(jacobian_layers, injection=injection)
    for injection in INJECTIONS
}
scale_configs = {
    scale: model_config(
        diagnostic_layers,
        gated_mlp=False,
        activation="relu",
        residual_scaling=scale,
        cycles=residual_scale_recurrences,
    )
    for scale in residual_scales
}
projection_config = model_config(
    diagnostic_layers,
    injection="projection",
    gated_mlp=False,
    activation="relu",
)

In [ ]:
def summarize_stability(run):
    """Keep token-averaged comparisons from the first recurrent layer to all finals."""
    layers = run.recurrent_layer_indices
    indices = [
        block.virtual_layer_index
        for block in run.per_virtual_layer_metrics
        if block.layer_index == layers[0]
    ]
    curves = {
        metric: getattr(run, f"to_final_state_residual_{metric}")[indices][:, layers]
        .float()
        .mean(-1)
        .cpu()
        for metric in ("cosine_sim", "diff_norm")
    }
    return dict(
        layers=layers,
        curves=curves,
        residual_norms=run.get_metric("residual_norms").float().cpu(),
        last_step_distance=run.successive_recurrence_residual_diff_norm[layers[-1]][-1]
        .float()
        .mean()
        .item(),
    )


@torch.no_grad()
def collect_stability(model_settings, run_seed, tokens):
    _set_seed(run_seed)
    bridge, structure = load_model(
        ModelSpec(model_settings),
        device=config.device,
        dtype=torch.float32,
    )
    _, cache = bridge.run_with_cache(
        tokens.to(config.device),
        accumulate=True,
        names_filter=structure.hook_names(),
        device="cpu",
    )
    run = compute_model_run_data(
        cache,
        tokens,
        structure,
        compute_pairwise=False,
        compute_differences=True,
    )[0]
    summary = summarize_stability(run)
    del run, cache, bridge
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary


def mean_std(values):
    return values.mean(0), values.std(0, correction=1 if len(values) > 1 else 0)


def reference_curves(examples, metric):
    values = torch.stack([example["curves"][metric] for example in examples])
    # Choose the worst final layer AFTER averaging tokens and seeds.
    mean = values.mean(0)
    worst = mean.argmin(-1) if metric == "cosine_sim" else mean.argmax(-1)
    worst_values = values[:, torch.arange(len(worst)), worst]
    return mean_std(values[:, :, 0]), mean_std(worst_values)

In [ ]:
@torch.no_grad()
def recurrent_jacobian(model, embeddings, initial_state):
    """First-token dF_x(z)/dz, with embeddings held fixed and no reinitialisation."""
    # Causal attention makes the first token independent of later tokens.
    x = embeddings[:, :1].detach()
    z = initial_state[0, 0].detach()
    with sdpa_kernel(SDPBackend.MATH):
        return torch.func.jacrev(
            lambda state: model.recurrent_step(state[None, None], x)[0, 0]
        )(z).cpu()


@torch.no_grad()
def measure_jacobian(model_settings, run_seed, tokens):
    _set_seed(run_seed)
    bridge, _ = load_model(
        ModelSpec(model_settings),
        device=config.device,
        dtype=torch.float32,
    )
    model = bridge.original_model
    x = model.embed_tokens(tokens.to(config.device))
    z = (
        torch.randn_like(x) * model_settings.initialization_variance
        if model_settings.input_injection
        else x.clone()
    )
    jacobian = recurrent_jacobian(model, x, z)
    return torch.linalg.eigvals(jacobian)


@torch.no_grad()
def measure_projection(model_settings, run_seed):
    _set_seed(run_seed)
    bridge, _ = load_model(
        ModelSpec(model_settings),
        device=config.device,
        dtype=torch.float32,
    )
    # The projection receives [embeddings, latent state]; retain its latent block.
    weight = bridge.original_model.input_proj.weight[:, width:].detach().float().cpu()
    std = weight.std().item()
    return {
        "spectral_norm": torch.linalg.matrix_norm(weight, ord=2).item(),
        "spectral_radius": torch.linalg.eigvals(weight).abs().max().item(),
        "weight_std": std,
        "iid_radius_estimate": std * np.sqrt(width),
    }

## Generate and reduce

Build every listed depth with its actual number of transformer layers. Each seed
uses the same random token IDs across configurations. The stability sweep uses
gated SiLU MLPs; the residual-scale and projection measurements use six-layer,
ungated ReLU models with Pre Norm. The Jacobian comparison uses one-layer Pre Norm
models, evaluated at their initial recurrent states.

Retain token-averaged curves, spectra, scalar measurements, and run settings in
`overview.pt`; scalar measurements also go to CSV. Release models and activation
caches after reduction. The cells below this section only use the saved summaries.

In [ ]:
settings = {
    "experiment": "initialisation",
    "dataset": "random_initialisation",
    "mode": config.mode,
    "seeds": seeds,
    "recurrences": recurrences,
    "summary_layers": summary_layers,
    "sequence_length": sequence_length,
    "device": config.device,
    "dtype": "float32",
    "stability_configs": [
        {"normalisation": norm, **asdict(model_settings)}
        for (_, norm, _), model_settings in stability_configs.items()
    ],
    "jacobian_configs": [asdict(c) for c in jacobian_configs.values()],
    "scale_configs": [asdict(c) for c in scale_configs.values()],
    "projection_config": asdict(projection_config),
    "versions": {
        name: version(name) for name in ("torch", "transformer-lens", "transformers")
    },
}
settings["cache"] = result_cache.settings
run_dir = save_results({}, settings, config.output_dir)
inputs = {
    run_seed: torch.randint(
        vocab_size,
        (1, sequence_length),
        generator=torch.Generator().manual_seed(run_seed),
    )
    for run_seed in seeds
}
stability = {}
for key, model_settings in tqdm(
    stability_configs.items(), desc="Stability configurations"
):
    stability[key] = [
        cached_measurement(
            collect_stability, model_settings, run_seed, inputs[run_seed]
        )
        for run_seed in seeds
    ]

In [ ]:
spectra, scale_measurements, projection_measurements = {}, [], []
for run_seed in tqdm(seeds, desc="Jacobian and scalar measurements"):
    for injection, model_settings in jacobian_configs.items():
        spectra.setdefault(injection, []).append(
            cached_measurement(
                measure_jacobian, model_settings, run_seed, inputs[run_seed]
            )
        )
    for scale, model_settings in scale_configs.items():
        summary = cached_measurement(
            collect_stability, model_settings, run_seed, inputs[run_seed]
        )
        scale_measurements.append(
            dict(
                seed=run_seed,
                residual_scale=scale,
                last_step_distance=summary["last_step_distance"],
            )
        )
    projection_measurements.append(
        dict(
            seed=run_seed,
            **cached_measurement(measure_projection, projection_config, run_seed),
        )
    )
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

scale_measurements = pd.DataFrame(scale_measurements)
projection_measurements = pd.DataFrame(projection_measurements)
scale_measurements.to_csv(run_dir / "residual_scaling.csv", index=False)
projection_measurements.to_csv(run_dir / "projection_measurements.csv", index=False)
torch.save(
    dict(
        settings=settings,
        inputs=inputs,
        stability=stability,
        spectra=spectra,
        residual_scaling=scale_measurements.to_dict("records"),
        projection=projection_measurements.to_dict("records"),
    ),
    run_dir / "overview.pt",
)
print(f"Saved {len(seeds)} seed(s) per configuration to {run_dir}")
result_cache.write_manifest(run_dir)

## What the measurements mean

Write $h_{r,l,t}$ for the output of layer $l$ in recurrence $r$ at token $t$.
The final reference is $h_{R,l,t}$ with **$R=128$** by default. It is an observed
endpoint, not a proven fixed point. Comparisons include recurrence $R$, whose
same-layer distance is zero and cosine similarity is one for nonzero states.
Layer outputs are measured before the optional end-of-cycle normalization.

| Measurement | Definition and interpretation |
| --- | --- |
| Same-layer similarity / distance | Cosine similarity / Euclidean distance from the **first recurrent layer** to its own final output. |
| Lowest similarity / largest distance | Compare that same source layer with every recurrent layer's final output, then select the least similar / most distant reference. The reference can vary with recurrence. |
| Residual norm | First-token L2 norm at every virtual layer, in forward order; this is not a token average. |
| Recurrent Jacobian | $J_z=\partial F_x(z)/\partial z$: one complete recurrence, holding the embeddings $x$ fixed. Evaluate at the initial state, with random $z_0$ for injection and $z_0=x$ otherwise. Plot the first-token block's eigenvalues; the unit circle marks magnitude one. |
| Residual-scale measurement | Mean token distance between recurrences 255 and 256, at the final layer of a six-layer ungated ReLU model. Scaling multiplies the latent state once per recurrence. |
| Projection measurements | Spectral norm, spectral radius, and weight standard deviation of $W_z$, the latent half of $W[x,z]$. |

For similarities and distances, **average tokens within each seed first**, then
average seeds. Select the worst reference using these averaged values; its band
is the standard deviation across seeds for that selected reference. Other bands
also show one sample standard deviation across seeds, or zero with one seed.
The comparison plots and summary use the first recurrent layer, while
residual norms show every virtual layer. Depth comparisons construct distinct
4-, 8-, 12-, and 16-layer models; the summary explicitly uses 12 layers.

The Jacobian differentiates the recurrent state, not the embeddings. It includes
input injection, residual scaling, every block, and end-of-cycle normalization.
First-token causality lets us compute the exact first-token block without forming
a sequence-sized Jacobian. Its spectrum describes that local block at
initialisation, not the full sequence or the endpoint's stability. Plot every
seed's spectrum without averaging unordered eigenvalues.

In [ ]:
NORM_COLORS = {"Pre": "C0", "Huginn-0125": "C2", "Ouro": "C3"}
METRIC_LABELS = {
    "cosine_sim": "Final-state cosine sim.",
    "diff_norm": "Final-state distance",
}


def plot_band(ax, x, stats, **style):
    mean, std = stats
    (line,) = ax.plot(x, mean, **style)
    ax.fill_between(x, mean - std, mean + std, color=line.get_color(), alpha=0.2)


def stability_axes(metric):
    """Normalization across columns; input injection across rows."""
    fig, axes = plt.subplots(
        2, 3, figsize=get_figsize(True, (2, 3), 1.5), layout="constrained"
    )
    for row, injection in enumerate(INJECTIONS):
        for col, norm in enumerate(NORMALISATIONS):
            suffix = " + input inj." if injection else ""
            axes[row, col].set(title=f"{norm} Norm{suffix}", xlabel="Recurrence")
        axes[row, 0].set_ylabel(METRIC_LABELS[metric])
    return fig, axes


def plot_stability(metric, filename):
    """Compare same-layer and worst-reference curves at the summary depth."""
    fig, axes = stability_axes(metric)
    worst_label = "Lowest sim." if metric == "cosine_sim" else "Largest dist."
    for row, injection in enumerate(INJECTIONS):
        for col, norm in enumerate(NORMALISATIONS):
            own, worst = reference_curves(
                stability[summary_layers, norm, injection], metric
            )
            x = np.arange(1, len(own[0]) + 1)
            plot_band(axes[row, col], x, own, color="C0", label="Same layer")
            plot_band(axes[row, col], x, worst, color="C1", label=worst_label)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside upper center", ncol=2, frameon=False)
    save_figure(fig, run_dir, filename, bbox_inches=None)


def plot_depth_dependence():
    """Compare same-layer difference norms, with one colour per block depth."""
    fig, axes = stability_axes("diff_norm")
    for row, injection in enumerate(INJECTIONS):
        for col, norm in enumerate(NORMALISATIONS):
            for index, depth in enumerate(layer_counts):
                examples = stability[depth, norm, injection]
                own = mean_std(
                    torch.stack([e["curves"]["diff_norm"][:, 0] for e in examples])
                )
                x = np.arange(1, len(own[0]) + 1)
                plot_band(
                    axes[row, col], x, own, color=f"C{index}", label=f"{depth} layers"
                )
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="outside upper center",
        ncol=len(layer_counts),
        frameon=False,
    )
    save_figure(
        fig,
        run_dir,
        "stability_initialised_to_final_state_residual_diff_norm_all_layers",
        bbox_inches=None,
    )

In [ ]:
def plot_summary(filename="summarised_stability"):
    fig, axes = plt.subplots(
        1, 3, figsize=get_figsize(True, (1, 3), 2), layout="constrained"
    )
    for injection in INJECTIONS:
        for norm in NORMALISATIONS:
            examples = stability[summary_layers, norm, injection]
            own = mean_std(
                torch.stack([e["curves"]["cosine_sim"][:, 0] for e in examples])
            )
            style = dict(color=NORM_COLORS[norm], linestyle="--" if injection else "-")
            x = np.arange(1, len(own[0]) + 1)
            plot_band(axes[0], x, own, **style)
            norms = mean_std(torch.stack([e["residual_norms"] for e in examples]))
            plot_band(axes[1], np.arange(1, len(norms[0]) + 1), norms, **style)
    axes[0].set(
        title="Same Layer", xlabel="Recurrence", ylabel=METRIC_LABELS["cosine_sim"]
    )
    axes[1].set(title="Residual Norms", xlabel="Virtual layer", ylabel="Residual norm")
    handles = [
        Line2D([], [], color=color, label=norm) for norm, color in NORM_COLORS.items()
    ]
    handles.append(Line2D([], [], color="black", linestyle="--", label="Input Inj."))
    fig.legend(handles=handles, loc="outside upper center", ncol=4, frameon=False)

    ax = axes[2]
    for injection in INJECTIONS:
        eigenvalues = torch.cat(spectra[injection]).numpy()
        ax.scatter(
            eigenvalues.real,
            eigenvalues.imag,
            s=5,
            alpha=0.6,
            color="black" if injection else "C4",
            marker="x" if injection else "o",
            label="Input Inj." if injection else "Residual",
        )
    ax.add_patch(plt.Circle((0, 0), 1, color="grey", fill=False, linestyle="--"))
    ax.axhline(0, color="grey", linestyle="--", linewidth=0.5)
    ax.axvline(0, color="grey", linestyle="--", linewidth=0.5)
    extent = max(
        1.5,
        1.05 * max(e.abs().max().item() for values in spectra.values() for e in values),
    )
    ax.set(
        title="Eigenvalues at Init",
        xlabel="Real part",
        ylabel="Imaginary part",
        xlim=(-extent, extent),
        ylim=(-extent, extent),
        aspect="equal",
    )
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.14), ncol=2, frameon=False)
    save_figure(fig, run_dir, filename, bbox_inches=None)

## Normalization and input injection

Full-width grids for the 12-layer models. Each panel compares the first recurrent
layer with its own final output and with the worst-matching final layer.

In [ ]:
plot_stability(
    "cosine_sim",
    "stability_initialised_to_final_state_residual_cosine_sim",
)

In [ ]:
plot_stability(
    "diff_norm",
    "stability_initialised_to_final_state_residual_diff_norm",
)

## Dependence on recurrent block depth

Compare the same-layer difference norm for all four block depths. Colours identify
4, 8, 12, and 16 layers; each model uses the first recurrent layer's own final output
as its reference. Bands show one standard deviation across seeds.

In [ ]:
plot_depth_dependence()

## Paper summary

Three full-width panels show same-layer final-reference cosine similarity,
residual magnitudes, and the recurrent Jacobian spectrum. The first two reuse the
12-layer models; the Jacobians use the separate one-layer models.

In [ ]:
plot_summary()

## Residual scaling and projection measurements

The first table reports the last successive-recurrence distance for each residual
scale, averaged over seeds. This diagnostic retains its longer, 256-recurrence
rollout; the final-reference stability curves above use 128 recurrences.

The projection table measures the square latent block $W_z\in\mathbb{R}^{d\times d}$
of the full $d\times 2d$ projection. The estimate $\sigma\sqrt{d}$ is an asymptotic
i.i.d.-weight spectral-radius heuristic, not a measured radius. The comparison
standards below distinguish the $2d$ fan-in of the projection from a $d$-wide layer.
PyTorch's default linear initialization is uniform, with standard deviation
$1/\sqrt{6d}$ here; the Glorot and He values are references, not applied initializers.

In [ ]:
scale_summary = scale_measurements.groupby("residual_scale")["last_step_distance"].agg(
    ["mean", "std"]
)
# A single seed has no spread; match the zero-width bands in quick mode.
display(scale_summary.fillna(0).rename_axis("Residual scale"))

In [ ]:
projection_summary = (
    projection_measurements.drop(columns="seed").agg(["mean", "std"]).T.fillna(0)
)
display(projection_summary)
initialization_references = pd.Series(
    {
        "PyTorch linear std (fan-in 2d)": 1 / np.sqrt(6 * width),
        "Glorot std (fan-in 2d, fan-out d)": np.sqrt(2 / (3 * width)),
        "He std (fan-in 2d)": np.sqrt(2 / (2 * width)),
        "He std (fan-in d)": np.sqrt(2 / width),
    },
    name="Standard deviation",
)
display(initialization_references.to_frame())
initialization_references.to_csv(run_dir / "initialization_references.csv")

In [ ]:
paper_dir = export_paper_figures(config, run_dir)
print(f"Run outputs: {run_dir}")
if paper_dir is not None:
    print(f"Paper figures: {paper_dir}")